In [1]:
import os
import sys
import json
import re
import uuid
from typing import List, Dict, Any, TypedDict, Annotated
import requests
import wikipediaapi
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from gigachat import GigaChat
from getpass import getpass
import time

c:\Users\mark\Desktop\AgenticRAG\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
YANDEX_WEATHER_KEY = getpass("API-ключ Яндекс Погоды: ")

In [2]:
ACCESS_TOKEN = getpass("GigaChat access token: ")

In [4]:
giga = GigaChat(
    access_token=ACCESS_TOKEN,
    verify_ssl_certs=False
)

In [5]:
try:
    response = giga.chat("Привет, как дела?")
    print("GigaChat работает:", response.choices[0].message.content[:100])
except Exception as e:
    print("Ошибка:", e)

GigaChat работает: Привет! Всё отлично, готов общаться, решать задачи или обсуждать любые темы. Чем займёмся?


In [8]:
def fetch_articles_from_categories(category_names: List[str], max_per_category: int = 300, total_limit: int = 2000) -> List[Dict[str, Any]]:
    """
    Загружает статьи из указанных категорий Википедии.
    """
    wiki = wikipediaapi.Wikipedia(language='ru', user_agent='AgenticRAG_Demo/1.0')
    all_docs = []
    seen_titles = set()

    for cat_name in category_names:
        cat_page = wiki.page(f"Категория:{cat_name}")
        if not cat_page.exists():
            print(f"Категория '{cat_name}' не найдена")
            continue
        members = cat_page.categorymembers
        count = 0
        for title, page in members.items():
            if page.ns != 0:
                continue
            if title in seen_titles:
                continue
            if count >= max_per_category:
                break

            content = page.text
            if content.strip():
                all_docs.append({
                    "content": content,
                    "metadata": {"title": title, "url": page.fullurl}
                })
                seen_titles.add(title)
                count += 1
            time.sleep(0.05)
        print(f"Из категории '{cat_name}' загружено {count} статей")
        if len(all_docs) >= total_limit:
            print(f"Достигнут лимит в {total_limit} статей")
            break
    return all_docs

categories = [
    "Наука",
    "Технологии",
    "Информатика",
    "Математика",
    "Физика",
    "Химия",
    "Биология",
    "География",
    "История",
    "Россия",
    "Культура",
    "Искусство",
    "Литература",
    "Философия",
    "Экономика",
    "Право",
    "Медицина",
    "Психология",
    "Спорт"
]

print("Загрузка статей из категорий...")
docs = fetch_articles_from_categories(
    categories,
    max_per_category=15,
    total_limit=150
)
print(f"Загружено {len(docs)} статей.")

Загрузка статей из категорий...
Из категории 'Наука' загружено 12 статей
Категория 'Технологии' не найдена
Из категории 'Информатика' загружено 15 статей
Из категории 'Математика' загружено 15 статей
Из категории 'Физика' загружено 15 статей
Из категории 'Химия' загружено 15 статей
Из категории 'Биология' загружено 15 статей
Из категории 'География' загружено 14 статей
Из категории 'История' загружено 15 статей
Из категории 'Россия' загружено 1 статей
Из категории 'Культура' загружено 15 статей
Из категории 'Искусство' загружено 15 статей
Из категории 'Литература' загружено 15 статей
Достигнут лимит в 150 статей
Загружено 162 статей.


In [9]:
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))
collection = chroma_client.get_or_create_collection(
    name="wikipedia_docs",
    embedding_function=None
)

ids = [str(uuid.uuid4()) for _ in docs]
contents = [doc["content"] for doc in docs]
metadatas = [doc["metadata"] for doc in docs]
embeddings = embedding_model.encode(contents, convert_to_numpy=True).tolist()
collection.add(
    ids=ids,
    documents=contents,
    embeddings=embeddings,
    metadatas=metadatas
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10031.66it/s]


In [10]:
def search_documents(query: str, top_k: int = 3) -> str:
    query_embedding = embedding_model.encode([query], convert_to_numpy=True).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    output = "Найденные документы:\n\n"
    for i, doc_text in enumerate(results["documents"][0], 1):
        meta = results["metadatas"][0][i-1] if results["metadatas"] else {}
        score = results["distances"][0][i-1] if results["distances"] else None
        title = meta.get("title", "Без названия")
        output += f"--- {i}. {title} (score: {score:.4f}) ---\n"
        output += doc_text[:500] + "...\n\n"
    return output.strip()

In [11]:
def calculator(expression: str) -> str:
    cleaned = re.sub(r'[^0-9+\-*/().% ]', '', expression)
    if not cleaned:
        return "Ошибка: пустое выражение."
    try:
        result = eval(cleaned, {"__builtins__": None}, {})
        return f"Результат: {result}"
    except Exception as e:
        return f"Ошибка вычисления: {str(e)}"

In [12]:
def get_coordinates(city_name: str) -> tuple:
    """
    Преобразует название города в координаты через Nominatim (OpenStreetMap)
    """
    headers = {
        "User-Agent": "AgenticRAG_Demo/1.0 (pet-project)"
    }
    url = "https://nominatim.openstreetmap.org/search"
    params = {
        "q": city_name,
        "format": "json",
        "limit": 1,
        "accept-language": "ru"
    }

    time.sleep(1)
    resp = requests.get(url, params=params, headers=headers, verify=False, timeout=5)
    resp.raise_for_status()

    data = resp.json()
    if not data:
        return None, None

    lat = float(data[0]["lat"])
    lon = float(data[0]["lon"])
    return lat, lon

In [13]:
def weather(city: str) -> str:
    lat, lon = get_coordinates(city)

    weather_url = "https://api.weather.yandex.ru/v2/forecast"
    params = {"lat": lat, "lon": lon, "lang": "ru_RU"}
    headers = {"X-Yandex-Weather-Key": YANDEX_WEATHER_KEY}

    resp = requests.get(weather_url, params=params, headers=headers, verify=False, timeout=5)
    resp.raise_for_status()
    data = resp.json()
    fact = data["fact"]
    temp = fact["temp"]
    condition = fact["condition"]
    wind = fact["wind_speed"]
    pressure = fact["pressure_mm"]

    condition_translation = {
        "clear": "ясно",
        "partly-cloudy": "переменная облачность",
        "cloudy": "облачно",
        "overcast": "пасмурно",
        "light-rain": "небольшой дождь",
        "rain": "дождь",
        "heavy-rain": "сильный дождь",
        "showers": "ливень",
        "wet-snow": "дождь со снегом",
        "light-snow": "небольшой снег",
        "snow": "снег",
        "heavy-snow": "сильный снег",
        "thunderstorm": "гроза"
    }
    condition_ru = condition_translation.get(condition, condition)

    return f"Погода в {city}: {condition_ru}, {temp}°C, ветер {wind} м/с, давление {pressure} мм рт. ст."

In [14]:
TOOL_REGISTRY = {
    "search": {
        "func": search_documents,
        "description": "Поиск по базе знаний (Wikipedia). Вход: поисковый запрос."
    },
    "calculator": {
        "func": calculator,
        "description": "Математические вычисления. Вход: выражение, например '2+2*3'."
    },
    "weather": {
        "func": weather,
        "description": "Погода в городе (демо). Вход: название города."
    }
}

def get_tool_descriptions() -> str:
    return "\n".join([f"- {name}: {info['description']}" for name, info in TOOL_REGISTRY.items()])

In [15]:
def plan_next_action(state: Dict) -> Dict:
    last_msg = state["messages"][-1]
    user_query = last_msg.content if hasattr(last_msg, 'content') else last_msg["content"]

    tool_results = state.get("tool_results", [])
    tool_results_text = "\n".join(tool_results) if tool_results else "Нет"

    prompt = f"""
Ты — планировщик агента. У тебя есть доступ к инструментам:
{get_tool_descriptions()}

Вопрос пользователя: {user_query}

Результаты предыдущих действий (если есть):
{tool_results_text}

Ты должен решить, что делать дальше. Ответь в формате JSON:
- Если нужен инструмент: {{"action": "search/calculator/weather", "tool_input": "аргумент"}}
- Если можно ответить сразу: {{"action": "final_answer", "answer": "твой ответ"}}

Только JSON, без пояснений.
"""
    system_prompt = "Ты — полезный ассистент-планировщик."

    response = giga.chat({
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]
    })
    raw = response.choices[0].message.content
    print("Планировщик:", raw)
    start = raw.find('{')
    end = raw.rfind('}') + 1
    if start != -1 and end != -1:
        plan = json.loads(raw[start:end])
    else:
        plan = json.loads(raw)

    if plan.get("action") != "final_answer":
        tool_name = plan.get("action")
        if tool_name in TOOL_REGISTRY:
            plan["tool_name"] = tool_name
            plan["tool_input"] = plan.get("tool_input", plan.get("input", ""))
            plan["action"] = "tool"
        else:
            if plan.get("action") == "tool" and plan.get("tool_name"):
                pass
            else:
                plan = {"action": "final_answer", "answer": "Извините, я не понял запрос."}

    state["current_plan"] = plan
    if plan.get("action") == "final_answer":
        state["messages"].append(AIMessage(content=plan.get("answer", "Ответ не сформулирован.")))
    return state

In [16]:
def execute_tool(state: Dict) -> Dict:
    plan = state.get("current_plan", {})
    if plan.get("action") != "tool":
        return state

    tool_name = plan.get("tool_name")
    tool_input = plan.get("tool_input")
    if tool_name in TOOL_REGISTRY:
        func = TOOL_REGISTRY[tool_name]["func"]
        result = func(tool_input)
        state["tool_results"].append(f"{tool_name}({tool_input}) -> {result}")
        state["messages"].append(ToolMessage(content=result, tool_call_id=tool_name))
    else:
        state["tool_results"].append(f"Неизвестный инструмент: {tool_name}")
    return state

In [17]:
def generate_final_answer(state: Dict) -> Dict:
    """Формирует финальный ответ на основе результатов инструментов."""
    last_msg = state["messages"][0]
    user_query = last_msg.content if hasattr(last_msg, 'content') else last_msg["content"]

    tool_results = state.get("tool_results", [])
    tool_results_text = "\n".join(tool_results) if tool_results else "Нет"

    prompt = f"""
Ты — ассистент. Пользователь задал вопрос: {user_query}

Были использованы инструменты и получены следующие результаты:
{tool_results_text}

На основе этих результатов дай развернутый и полезный ответ пользователю.
Если информации недостаточно, честно скажи об этом.
"""
    system_prompt = "Ты — полезный и точный ассистент."

    response = giga.chat({
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]
    })
    answer = response.choices[0].message.content
    state["messages"].append(AIMessage(content=answer))
    return state

In [18]:
class AgentState(TypedDict):
    messages: Annotated[List[Any], add_messages]
    current_plan: Dict
    tool_results: List[str]
    iteration: int
    max_iterations: int

def should_continue(state: AgentState) -> str:
    if state["iteration"] >= state["max_iterations"]:
        return "end"
    plan = state.get("current_plan", {})
    if plan.get("action") == "final_answer":
        return "end"
    if plan.get("action") == "tool":
        return "generate"
    return "end"

workflow = StateGraph(AgentState)

workflow.add_node("planner", plan_next_action)
workflow.add_node("executor", execute_tool)
workflow.add_node("generator", generate_final_answer)

workflow.set_entry_point("planner")

workflow.add_conditional_edges(
    "planner",
    should_continue,
    {
        "generate": "executor",
        "end": END
    }
)
workflow.add_edge("executor", "generator")
workflow.add_edge("generator", END)

app = workflow.compile()

In [19]:
def ask_agent(query: str, max_iter: int = 3) -> str:
    initial_state = {
        "messages": [HumanMessage(content=query)],
        "current_plan": {},
        "tool_results": [],
        "iteration": 0,
        "max_iterations": max_iter
    }
    final_state = app.invoke(initial_state)
    for msg in reversed(final_state["messages"]):
        if isinstance(msg, AIMessage):
            return msg.content
    return "Ответ не получен."

In [24]:
test_questions = [
    "Что такое класс в питоне?",
    "Сколько будет 25 * 4 + 10?",
    "Какая погода в Москве?"
]

for q in test_questions:
    print(f"\nВопрос: {q}")
    answer = ask_agent(q, max_iter=3)
    print(f"Ответ: {answer}")
    time.sleep(1)


Вопрос: Что такое класс в питоне?
Планировщик: {"action": "search", "tool_input": "класс в питоне"}
Ответ: Полученная информация не содержит прямых ответов на вопрос о том, что такое "класс" в Python. Однако можно самостоятельно предоставить необходимую информацию на основе общих знаний о программировании на Python.

### Класс в Python

В языке программирования Python класс является основой объектно-ориентированного программирования (ООП). Класс определяет шаблон (или структуру) объектов, которые будут созданы на его основе. 

#### Основные характеристики класса:

1. **Определение структуры данных**: Класс описывает атрибуты (переменные) и методы (функции), которыми будет обладать каждый экземпляр объекта этого класса.
   
   ```python
   class Person:
       def __init__(self, name, age):
           self.name = name
           self.age = age

       def greet(self):
           print(f"Hello, my name is {self.name} and I am {self.age} years old.")
   ```

2. **Создание экземпляров (об

c:\Users\mark\Desktop\AgenticRAG\myenv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'nominatim.openstreetmap.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\mark\Desktop\AgenticRAG\myenv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.weather.yandex.ru'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Ответ: Погода в Москве на текущий момент следующая:

- Облачность: облачно
- Температура воздуха: 18°C
- Ветер: его скорость составляет примерно 2.8 метра в секунду
- Атмосферное давление: 738 миллиметров ртутного столба

Эта информация актуальна на момент запроса. Для получения наиболее точного и актуального прогноза погоды рекомендуется воспользоваться специализированными метеосайтами или приложениями.
